# Analyse exploratoire (EDA) — charge horaire PJM

**Objectif :**
Comprendre la série temporelle avant la modélisation par régression polynomiale : tendance globale, saisonnalité, profils jour/nuit, valeurs manquantes ou aberrantes.

**Contexte :**
Projet de modélisation non linéaire socio-énergétique visant à analyser la relation entre la consommation électrique et les dynamiques temporelles.

**Données attendues :**

* Fichier CSV :
socio-energy-model/data/raw/

* Exemple :
PJME_hourly.csv

**Convention de nommage des notebooks :**

* Format :
{ordre}_{type}_{sujet}.ipynb

* Exemple :
01_eda_pjm_hourly_load.ipynb

* Règle :
L’ordre suit le pipeline du projet (EDA → régression polynomiale → évaluation).

**Convention de structure des chemins :**

* Notebook :
notebooks/

* Racine projet :
socio-energy-model/

* Données brutes :
data/raw/

* Données traitées :
data/processed/

**Remarque :**

* Cette étape d’EDA permet d’identifier les relations non linéaires dans la consommation énergétique, justifiant l’utilisation de la régression polynomiale pour la modélisation.

In [1]:
from pathlib import Path

# Racine du sous-projet energy_forecast (dossier au-dessus de notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"

print("Racine projet :", ROOT.resolve())
print("Fichiers CSV dans data/raw :", sorted(p.name for p in DATA_RAW.glob("*.csv")))

Racine projet : C:\Users\ISMAELLE\Downloads\socio-energy-polynomial-model\socio-energy-model
Fichiers CSV dans data/raw : ['PJME_hourly.csv']


In [2]:
import pandas as pd


candidates = ["PJME_hourly.csv", "PJM_Load_hourly.csv"]
csv_paths = [DATA_RAW / name for name in candidates if (DATA_RAW / name).exists()]
if not csv_paths:
    others = sorted(DATA_RAW.glob("*.csv"))
    if not others:
        raise FileNotFoundError(f"Aucun CSV dans {DATA_RAW.resolve()} — copier un fichier .csv dans data/raw/.")
    csv_paths = [others[0]]

CSV_PATH = csv_paths[0]
print("Fichier chargé :", CSV_PATH.name)

df = pd.read_csv(CSV_PATH, parse_dates=["Datetime"])
df = df.sort_values("Datetime").reset_index(drop=True)

display(df.head())
print(df.dtypes)
print("Lignes :", len(df), "| Plage :", df["Datetime"].min(), "→", df["Datetime"].max())
print("NaN par colonne :\n", df.isna().sum())

Fichier chargé : PJME_hourly.csv


,Datetime,PJME_MW
0,2002-01-01 01:00:00,30393.0
1,2002-01-01 02:00:00,29265.0
2,2002-01-01 03:00:00,28357.0
3,2002-01-01 04:00:00,27899.0
4,2002-01-01 05:00:00,28057.0


Datetime    datetime64[us]
PJME_MW            float64
dtype: object
Lignes : 145366 | Plage : 2002-01-01 01:00:00 → 2018-08-03 00:00:00
NaN par colonne :
 Datetime    0
PJME_MW     0
dtype: int64


## Nettoyage pour la suite du projet (série régulière 1 h)

Les **doublons** à l’heure répétée (ex. `02:00` en novembre) et les **trous** d’une heure (ex. mars) viennent du **changement d’heure (DST)** en heure locale US, pas d’une erreur de fichier.

**Règles appliquées ci-dessous (à citer dans le rapport) :**

1. **Doublons** : pour un même `Datetime`, on prend la **moyenne** de la charge (les deux mesures correspondent à la même heure civile ambiguë).
2. **Grille complète** : on réindexe de la première à la dernière date avec `freq="h"`.
3. **Trous** : **interpolation linéaire** sur la colonne MW (typiquement le saut d’une heure au printemps), puis `ffill` / `bfill` de secours si une extrémité restait NaN.

Le fichier nettoyé est écrit dans `data/processed/` (ex. `PJME_hourly_clean.csv`) pour que le **prétraitement** et les **fenêtres 168→24** utilisent tous la même série sans doublon ni lacune.

In [7]:
# Nettoyage reproductible (DST) → série strictement horaire, une valeur par pas

DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

mw_cols = [c for c in df.columns if c.endswith("_MW")]
value_col = mw_cols[0] if mw_cols else [c for c in df.columns if c != "Datetime"][0]

# 1) Une seule ligne par horodatage : moyenne si doublon (heure ambiguë fin DST)
df_agg = (
    df.groupby("Datetime", as_index=False, observed=True)[value_col]
    .mean()
    .sort_values("Datetime")
    .reset_index(drop=True)
)

# 2) Grille horaire complète entre min et max
full_ix = pd.date_range(df_agg["Datetime"].min(), df_agg["Datetime"].max(), freq="h")
df_clean = df_agg.set_index("Datetime").reindex(full_ix)
df_clean.index.name = "Datetime"
df_clean = df_clean.reset_index()

# 3) Remplir les trous (saut d’heure printemps, etc.)
df_clean[value_col] = df_clean[value_col].interpolate(method="linear", limit_direction="both")
if df_clean[value_col].isna().any():
    df_clean[value_col] = df_clean[value_col].ffill().bfill()

assert df_clean["Datetime"].is_monotonic_increasing
assert not df_clean["Datetime"].duplicated().any()
assert not df_clean[value_col].isna().any()

out_path = DATA_PROCESSED / f"{CSV_PATH.stem}_clean.csv"
df_clean.to_csv(out_path, index=False)

print("Fichier sauvegardé :", out_path.resolve())
print("Lignes : brut =", len(df), "| après agrégation doublons =", len(df_agg), "| grille 1 h =", len(df_clean))
print("Doublons restants :", int(df_clean["Datetime"].duplicated().sum()), "| NaN MW :", int(df_clean[value_col].isna().sum()))

Fichier sauvegardé : C:\Users\ISMAELLE\Downloads\socio-energy-polynomial-model\socio-energy-model\data\processed\PJME_hourly_clean.csv
Lignes : brut = 145366 | après agrégation doublons = 145362 | grille 1 h = 145392
Doublons restants : 0 | NaN MW : 0
